# Phase 4.5 — Model Refinements

This notebook refines `04_models.ipynb` (which stays unchanged) in two parts:

**Primary analysis** — same research question, better methodology:
1. An **EDA scan of every candidate column** before any modeling decision.
2. The plan's **70/15/15 split** (train / validation / test) so tuning decisions never touch the test set.
3. **Tuning the Random Forest's depth** on the validation set — the default forest was overfitting.

**Extension analysis** — a follow-up question the primary results raise:
our three research-question factors explain ~40% of sleep quality. Is that a limit of the *data*
or of our *factors*? We add the rest of a person's lifestyle/context and measure the difference.
The research question itself does not change — the extension exists to explain the primary
result's ceiling.

## 0. EDA — look at every column before using any column

Rule: no variable enters a model without being examined first. The correlation scan below is also
our **selection evidence** — it justifies which columns we use, and which we exclude and why:

- **Leakage (excluded):** `cognitive_performance_score`, `felt_rested`, `sleep_disorder_risk` —
  these are *consequences* of sleeping well. Predicting sleep quality from them is circular.
- **Sleep measurements (excluded):** duration, REM %, deep-sleep %, latency, wake episodes,
  sleep-aid use — they describe the night itself. A lifestyle model must only use what a person
  knows *before* sleeping.
- **No signal (excluded):** `person_id` (random ID), `country`, `room_temperature_celsius`,
  `heart_rate_resting_bpm` (|r| < 0.07).

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, r2_score, classification_report

RANDOM_STATE = 117
df = pd.read_csv("../sleep_health_dataset.csv")

# correlation of every numeric column with the target, strongest first
corr = (df.select_dtypes("number").corr()["sleep_quality_score"]
          .drop("sleep_quality_score").sort_values(key=abs, ascending=False))
print(corr.round(3))

cognitive_performance_score    0.860
sleep_duration_hrs             0.646
stress_score                  -0.639
felt_rested                    0.479
wake_episodes_per_night       -0.381
work_hours_that_day           -0.369
sleep_latency_mins            -0.293
shift_work                    -0.263
rem_percentage                 0.254
deep_sleep_percentage          0.091
alcohol_units_before_bed      -0.081
screen_time_before_bed_mins   -0.079
heart_rate_resting_bpm        -0.066
bmi                           -0.057
caffeine_mg_before_bed        -0.050
sleep_aid_used                 0.049
age                           -0.025
room_temperature_celsius      -0.023
exercise_day                   0.022
nap_duration_mins             -0.016
steps_that_day                 0.014
weekend_sleep_diff_hrs        -0.006
person_id                      0.002
Name: sleep_quality_score, dtype: float64


Reading the scan: `stress_score` (−0.64) dominates as Phase 2 found. The two strongest
remaining columns — `cognitive_performance_score` (0.86) and `felt_rested` (0.48) — are exactly
the leakage columns we must refuse. Among legitimate pre-sleep columns, `work_hours_that_day`
(−0.37) and `shift_work` (−0.26) stand out; age and steps remain ~0.

In [2]:
# value sanity-check for the columns the extension will add
NEW_COLS = ["work_hours_that_day", "shift_work", "caffeine_mg_before_bed",
            "alcohol_units_before_bed", "screen_time_before_bed_mins",
            "bmi", "nap_duration_mins"]
print(df[NEW_COLS].describe().round(2))
print()
for c in ["chronotype", "mental_health_condition", "day_type", "season"]:
    print(c, "->", dict(df[c].value_counts()))

       work_hours_that_day  shift_work  caffeine_mg_before_bed  \
count            100000.00   100000.00               100000.00   
mean                  7.13        0.08                   38.85   
std                   3.48        0.28                   69.40   
min                   0.00        0.00                    0.00   
25%                   4.70        0.00                    0.00   
50%                   7.40        0.00                    0.00   
75%                   9.70        0.00                   80.00   
max                  18.00        1.00                  400.00   

       alcohol_units_before_bed  screen_time_before_bed_mins        bmi  \
count                 100000.00                    100000.00  100000.00   
mean                       0.60                        63.54      26.29   
std                        1.06                        44.55       4.48   
min                        0.00                         2.00      16.00   
25%                        0.0

All ranges are plausible (no impossible values, nothing missing), matching Phase 1's
verification of the original columns. The categorical columns have clean, small value sets
suitable for one-hot encoding.

## 1. The 70/15/15 split

| Slice | Size | Job |
|---|---|---|
| **Train** | 70% (70,000) | models learn here, and only here |
| **Validation** | 15% (15,000) | compare models, tune settings — we may look many times |
| **Test** | 15% (15,000) | touched **once** per final model, for the honest numbers |

We split the whole dataframe once so every experiment uses the same people, and **stratify** on
the quality bucket so all three slices keep the same 42/44/14 class mix.

In [3]:
# One shared bucket definition (same as Phase 3 histogram & 04_models):
# round the 1-10 score to the nearest integer, then Low <=4 | Medium 5-6 | High >=7
def to_bucket(scores):
    r = scores.round()
    return pd.cut(r, bins=[-np.inf, 4, 6, np.inf], labels=["Low", "Medium", "High"])

df["quality_bucket"] = to_bucket(df["sleep_quality_score"])

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=RANDOM_STATE,
                                     stratify=df["quality_bucket"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=RANDOM_STATE,
                                   stratify=temp_df["quality_bucket"])
print(f"train {len(train_df):,} | validation {len(val_df):,} | test {len(test_df):,}")

baseline_val = val_df["quality_bucket"].value_counts(normalize=True).max()
print(f"majority-class baseline (always guess 'Medium'): {baseline_val:.1%}")

train 70,000 | validation 15,000 | test 15,000
majority-class baseline (always guess 'Medium'): 44.5%


## 2. Fixing the Random Forest — depth tuning on the validation set

Why the default forest underperforms: unlimited-depth trees keep splitting until they have
memorized the training set — including pure noise in `steps_that_day` (thousands of distinct
values, ~zero real signal). Memorized noise doesn't transfer to new people. A depth cap forces
each tree to keep only patterns strong enough to repeat. We choose the cap with the *validation*
set — exactly what a validation set is for.

In [4]:
FEATURES = ["age", "stress_score", "steps_that_day", "exercise_day"]
TARGET = "sleep_quality_score"

X_train, b_train = train_df[FEATURES], train_df["quality_bucket"]
X_val, b_val = val_df[FEATURES], val_df["quality_bucket"]

for depth in [None, 4, 6, 8, 12]:
    rf = RandomForestClassifier(n_estimators=200, max_depth=depth,
                                random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, b_train)
    print(f"max_depth={str(depth):>4}   validation accuracy {accuracy_score(b_val, rf.predict(X_val)):.1%}")

max_depth=None   validation accuracy 54.5%


max_depth=   4   validation accuracy 61.6%


max_depth=   6   validation accuracy 61.8%


max_depth=   8   validation accuracy 61.8%


max_depth=  12   validation accuracy 61.3%


Unlimited depth loses ~7 points to a simple cap; depths 6–8 tie at the top and 12 starts
memorizing again. We choose **`max_depth=8`**.

**This upgrades the Phase 4 conclusion.** `04_models.ipynb` reported "the simpler model wins by 6
points." The fairer statement: **properly tuned, both models tie at ~62% — evidence the signal
really is one linear stress effect, with no hidden non-linear structure for the forest to find.**
Testing for extra structure and finding none is a result, not a failure.

## 3. Extension — why is the primary ceiling ~62%?

The primary analysis answers the research question: age, stress, and activity predict sleep
quality at ~62%, almost entirely through stress. The natural follow-up: **is 62% the limit of the
dataset, or of our three factors?**

To answer it we add every legitimate pre-sleep column that survived the EDA scan — lifestyle
behaviors (work hours, shift work, caffeine, alcohol, screen time, naps) plus personal context
(BMI, chronotype, mental-health condition, day type, season, gender, occupation). Categorical
columns are one-hot encoded. This is an **extension**, not a new research question: the primary
models above remain the project's answer.

In [5]:
TIER2_EXTRA = ["work_hours_that_day", "shift_work", "caffeine_mg_before_bed",
               "alcohol_units_before_bed", "screen_time_before_bed_mins",
               "bmi", "nap_duration_mins", "chronotype", "mental_health_condition",
               "day_type", "season", "gender", "occupation"]

X_all = pd.get_dummies(df[FEATURES + TIER2_EXTRA], drop_first=True)
tier1_cols = [c for c in X_all.columns if any(c.startswith(f) for f in FEATURES)]
tier2_cols = list(X_all.columns)

Xtr2, Xva2, Xte2 = (X_all.loc[d.index] for d in (train_df, val_df, test_df))
y_train, y_val, y_test = (d[TARGET] for d in (train_df, val_df, test_df))
b_test = test_df["quality_bucket"]

for name, cols in [("Primary (research question)", tier1_cols),
                   ("Extension (+ lifestyle/context)", tier2_cols)]:
    lin = LinearRegression().fit(Xtr2[cols], y_train)
    acc_lin = accuracy_score(b_val, to_bucket(pd.Series(lin.predict(Xva2[cols]), index=Xva2.index)))
    rf = RandomForestClassifier(n_estimators=200, max_depth=8,
                                random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr2[cols], b_train)
    acc_rf = accuracy_score(b_val, rf.predict(Xva2[cols]))
    gb = HistGradientBoostingClassifier(random_state=RANDOM_STATE).fit(Xtr2[cols], b_train)
    acc_gb = accuracy_score(b_val, gb.predict(Xva2[cols]))
    r2 = r2_score(y_val, lin.predict(Xva2[cols]))
    print(f"{name:<32} LinReg {acc_lin:.1%} | RF(d8) {acc_rf:.1%} | GradBoost {acc_gb:.1%} | R2 {r2:.3f}")

Primary (research question)      LinReg 62.0% | RF(d8) 61.8% | GradBoost 61.8% | R2 0.417


Extension (+ lifestyle/context)  LinReg 68.8% | RF(d8) 67.9% | GradBoost 69.1% | R2 0.612


Gradient Boosting is included as a *challenger*: if it could not beat our two models on the
primary features (it can't — all three tie at ~62%), that confirms the ceiling belongs to the
features, not the algorithms.

## 4. What the extension discovered

Because this is a multiple (OLS) regression, each coefficient is that variable's effect **holding
all the others constant** — e.g., the shift-work effect below compares people with the *same*
stress, age, and activity:

In [6]:
lin2 = LinearRegression().fit(Xtr2[tier2_cols], y_train)
coefs = pd.Series(lin2.coef_, index=tier2_cols)
top = coefs.reindex(coefs.abs().sort_values(ascending=False).index)[:10]
print("Largest effects on predicted sleep quality (points on the 1-10 scale):")
print(top.round(3))

Largest effects on predicted sleep quality (points on the 1-10 scale):
shift_work                        -1.004
mental_health_condition_Healthy    0.887
day_type_Weekend                   0.662
mental_health_condition_Both      -0.590
occupation_Freelancer              0.575
occupation_Retired                 0.527
occupation_Homemaker               0.523
occupation_Teacher                 0.420
stress_score                      -0.406
chronotype_Morning                 0.321
dtype: float64


Readable version:

- **Shift work costs ~1 full point** of sleep quality — the largest single lifestyle effect.
- Being mentally **healthy adds ~0.9 points** vs. having anxiety/depression ("Both" costs another ~0.6).
- **Weekends add ~0.7 points** vs. workdays.
- Occupation shifts the score about half a point; morning chronotypes sleep slightly better.
- Stress remains the biggest overall factor: −0.41 per point across a 9-point scale ≈ **3.7-point
  total swing**, larger than any single effect above.
- Age and steps stay ~0 **even after controlling for everything else** — the strongest version yet
  of our "age and activity don't matter" finding.

## 5. Final honest numbers — one visit to the test set

All decisions above were made on validation data only. Now, once, the held-out test set:

In [7]:
final = []
baseline_test = b_test.value_counts(normalize=True).max()
final.append(("Baseline (always Medium)", baseline_test, None))

lin1 = LinearRegression().fit(Xtr2[tier1_cols], y_train)
pred = to_bucket(pd.Series(lin1.predict(Xte2[tier1_cols]), index=Xte2.index))
final.append(("Primary - Linear Regression", accuracy_score(b_test, pred),
              r2_score(y_test, lin1.predict(Xte2[tier1_cols]))))

rf1 = RandomForestClassifier(n_estimators=200, max_depth=8,
                             random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr2[tier1_cols], b_train)
final.append(("Primary - Random Forest (d=8)", accuracy_score(b_test, rf1.predict(Xte2[tier1_cols])), None))

pred2 = to_bucket(pd.Series(lin2.predict(Xte2[tier2_cols]), index=Xte2.index))
final.append(("Extension - Linear Regression", accuracy_score(b_test, pred2),
              r2_score(y_test, lin2.predict(Xte2[tier2_cols]))))

gb2 = HistGradientBoostingClassifier(random_state=RANDOM_STATE).fit(Xtr2[tier2_cols], b_train)
final.append(("Extension - Gradient Boosting", accuracy_score(b_test, gb2.predict(Xte2[tier2_cols])), None))

table = pd.DataFrame(final, columns=["model", "test accuracy", "test R2"]).set_index("model")
table["lift over baseline"] = table["test accuracy"] - baseline_test
print(table.round(3))

                               test accuracy  test R2  lift over baseline
model                                                                    
Baseline (always Medium)               0.445      NaN               0.000
Primary - Linear Regression            0.612    0.411               0.167
Primary - Random Forest (d=8)          0.613      NaN               0.168
Extension - Linear Regression          0.683    0.601               0.238
Extension - Gradient Boosting          0.688      NaN               0.243


In [8]:
print("Per-class breakdown for the best extension model (Gradient Boosting):\n")
print(classification_report(b_test, gb2.predict(Xte2[tier2_cols]), labels=["Low","Medium","High"]))

Per-class breakdown for the best extension model (Gradient Boosting):

              precision    recall  f1-score   support

         Low       0.76      0.73      0.74      6228
      Medium       0.63      0.73      0.68      6674
        High       0.70      0.44      0.54      2098

    accuracy                           0.69     15000
   macro avg       0.70      0.63      0.65     15000
weighted avg       0.69      0.69      0.68     15000



## Conclusions

1. **Primary (answers the research question):** age, stress, and activity predict sleep quality at
   ~61% vs. a 44.5% baseline — almost entirely through stress. This ceiling is real: three
   different algorithms tie there.
2. **A fairly tuned Random Forest ties Linear Regression** — Phase 4's "simple model wins by 6"
   was an untuned-forest artifact.
3. **Extension:** the rest of a person's lifestyle/context lifts accuracy to ~68–69% (R² 0.41 →
   0.60), with new controlled effects: shift work −1.0, mental health ~0.9, weekends ~0.7.
4. **~40% of sleep quality remains unexplained by lifestyle entirely** — an honest limit to state
   on the slides.
5. All conclusions describe **synthetic data** — patterns built into this Kaggle dataset, not
   verified facts about human sleep. Validating on real-world data is the top future-work item.